In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from pathlib import Path

# 设置字体为Times New Roman
plt.rcParams['font.family'] = 'Arial'

# 读取可行性分析结果数据
data_path = Path("../result/island_viability_summary_electric.csv")
df = pd.read_csv(data_path)

# 显示数据概况
print(f"数据形状: {df.shape}")
print(f"情景列表: {df['scenario'].unique()}")
print(f"Viability gap统计:")
print(df['viability_gap'].describe())

In [ ]:
import pandas as pd
import numpy as np

# Read the data files
df_2050 = pd.read_csv(r'c:\Users\18130\Desktop\island_optimization_newver\result\island_cost_summary_2050.csv')
df_future_2050 = pd.read_csv(r'c:\Users\18130\Desktop\island_optimization_newver\result\island_cost_summary_future_2050.csv')

# Calculate total cost for each scenario
df_2050['total_cost'] = (df_2050['renewable_cost_per_capita'] + df_2050['storage_cost_per_capita'] +
                        df_2050['lng_cost_per_capita'] + df_2050['other_equipment_cost_per_capita'] +
                        df_2050['discard_cost_per_capita'] + df_2050['load_shedding_cost_per_capita'])

df_future_2050['total_cost'] = (df_future_2050['renewable_cost_per_capita'] + df_future_2050['storage_cost_per_capita'] + 
                                df_future_2050['lng_cost_per_capita'] + df_future_2050['other_equipment_cost_per_capita'] +
                                df_future_2050['discard_cost_per_capita'] +df_future_2050['load_shedding_cost_per_capita'])

# Calculate cost per capita (cost recovery electricity price)
df_2050['cost_per_capita'] = df_2050['total_cost'] 
df_future_2050['cost_per_capita'] = df_future_2050['total_cost'] 

# Merge the datasets on lat and lon to ensure we're comparing the same islands
merged_df = pd.merge(df_2050[['lat', 'lon', 'cost_per_capita']],
                    df_future_2050[['lat', 'lon', 'cost_per_capita']],
                    on=['lat', 'lon'],
                    suffixes=('_2050', '_future_2050'))

print(f'Total islands matched: {len(merged_df)}')

# Calculate percentage decrease for each island
# Percentage decrease = (original - new) / original * 100
merged_df['percentage_decrease'] = ((merged_df['cost_per_capita_2050'] -
merged_df['cost_per_capita_future_2050']) / merged_df['cost_per_capita_2050']) * 100

# Remove any infinite or NaN values
valid_data = merged_df[np.isfinite(merged_df['percentage_decrease'])]
print(f'Valid data points: {len(valid_data)}')

# Calculate statistics
mean_decrease = valid_data['percentage_decrease'].mean()
median_decrease = valid_data['percentage_decrease'].median()
std_decrease = valid_data['percentage_decrease'].std()

print(f'\n=== 成本回收电价下降统计 ===')
print(f'平均下降百分比: {mean_decrease:.4f}%')
print(f'中位数下降百分比: {median_decrease:.4f}%')
print(f'标准差: {std_decrease:.4f}%')
print(f'最大下降: {valid_data["percentage_decrease"].max():.4f}%')
print(f'最小下降: {valid_data["percentage_decrease"].min():.4f}%')

# Show some examples
print(f'\n=== 前10个岛屿的详细数据 ===')
sample_data = valid_data[['lat', 'lon', 'cost_per_capita_2050', 'cost_per_capita_future_2050',      
'percentage_decrease']].head(10)
for i, row in sample_data.iterrows():
    print(f'岛屿 ({row["lat"]:.6f}, {row["lon"]:.6f}): {row["cost_per_capita_2050"]:.2f} → {row["cost_per_capita_future_2050"]:.2f} (下降 {row["percentage_decrease"]:.4f}%)')

print(f'\n答案: output_future_2050情景相对于output_2050情景，所有岛屿成本回收电价的平均下降百分比为 {mean_decrease:.4f}%')

In [ ]:
# 导入必要的库
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import pandas as pd
import numpy as np
import seaborn as sns 
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import stats
import matplotlib.colors # Added for background colormap

# --- Nature 风格图表设置 ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 20,
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 18,
    'figure.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

# <<< MODIFICATION START: Added Six-Region Definitions and Functions >>>
# --- 六区域定义 ---
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",      # 区域1
    "Feasible\nHigh Cost\nHigh Affordability",     # 区域2
    "Feasible\nLow Cost\nLow Affordability",       # 区域3
    "Infeasible\nHigh Cost\nHigh Affordability",   # 区域4
    "Infeasible\nLow Cost\nLow Affordability",     # 区域5
    "Infeasible\nHigh Cost\nLow Affordability",    # 区域6
]

# --- 六区域颜色定义 ---
SIX_REGION_COLORS = {
    SIX_REGION_NAMES[0]: '#012A61', # 深蓝 (最理想)
    SIX_REGION_NAMES[1]: '#0B75B3', # 中蓝
    SIX_REGION_NAMES[2]: '#89CAEA', # 浅蓝 (勉强可行)
    SIX_REGION_NAMES[3]: "#F0D2D2", # 浅红 (成本问题)
    SIX_REGION_NAMES[4]: '#DC5654', # 中红 (支付能力问题)
    SIX_REGION_NAMES[5]: '#982B2D', # 深红 (最棘手)
}

# --- 核心分类函数 ---
def classify_point(x, y, median_breakeven):
    """根据成本中位数对单个点(x,y)进行六区域分类"""
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven
    is_feasible = y >= x
    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[0]
        elif not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[1]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[2]
        else: return SIX_REGION_NAMES[5]
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[3]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[4]
        elif not is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[5]
        else: return SIX_REGION_NAMES[3]

# --- 背景绘制函数 ---
def draw_six_region_backgrounds(ax, median_breakeven, xlim, ylim):
    """使用网格化和 contourf 的方法精确绘制六个区域的背景色。"""
    x_min, x_max = xlim
    y_min, y_max = ylim
    
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 500),
        np.linspace(y_min, y_max, 500)
    )
    
    region_to_id = {name: i for i, name in enumerate(SIX_REGION_NAMES)}
    
    vectorized_classify = np.vectorize(classify_point)
    grid_regions = vectorized_classify(xx, yy, median_breakeven)
    
    Z = np.zeros(grid_regions.shape)
    for name, region_id in region_to_id.items():
        Z[grid_regions == name] = region_id
        
    colors = [SIX_REGION_COLORS[name] for name in SIX_REGION_NAMES]
    cmap = matplotlib.colors.ListedColormap(colors)
    
    levels = np.arange(-0.5, len(colors), 1)
    ax.contourf(xx, yy, Z, levels=levels, cmap=cmap, alpha=0.5, zorder=0)

# <<< MODIFICATION END: Added Six-Region Definitions and Functions >>>


# --- 数据处理函数 (Original functions kept) ---
def assign_ipcc_region(lat, lon, ipcc_regions_gdf):
    """将岛屿坐标分配到IPCC区域"""
    try:
        point = Point(lon, lat)
        possible_matches_index = list(ipcc_regions_gdf.sindex.intersection(point.bounds))
        possible_matches = ipcc_regions_gdf.iloc[possible_matches_index]
        precise_matches = possible_matches[possible_matches.contains(point)]
        if not precise_matches.empty:
            return precise_matches.iloc[0]['Acronym']
        return 'Unknown'
    except:
        return 'Unknown'

def calculate_position_change(df_base, df_compare):
    """计算两个情景之间的位置变化"""
    merged = pd.merge(df_base[['island_id', 'tariff_breakeven']],
                      df_compare[['island_id', 'tariff_breakeven']],
                      on='island_id', suffixes=('_base', '_compare'))
    merged['position_change'] = merged['tariff_breakeven_compare'] - merged['tariff_breakeven_base']
    return merged

# <<< MODIFICATION START: Replaced statistics function with 6-region version >>>
# --- 修正的统计分析与绘图函数 (六区域版本) ---
def analyze_and_plot_statistics(df_analysis, scenario_config, median_breakeven, label_mapping):
    """
    对数据进行六区域分类，并根据提供的 label_mapping 绘制 Q1-Q6 条形图。
    """
    results = []
    
    ordered_q_labels = sorted(list(set(label_mapping.values())))

    # 1. 遍历每个情景进行统计
    for scenario_name, config in scenario_config.items():
        scenario_df = df_analysis[df_analysis['scenario'] == scenario_name].copy()
        if scenario_df.empty:
            continue

        # 应用六区域分类
        scenario_df['region_class'] = scenario_df.apply(
            lambda row: classify_point(row['tariff_breakeven'], row['tariff_affordable'], median_breakeven),
            axis=1
        )
        
        counts = scenario_df['region_class'].value_counts().to_dict()
        
        # 使用映射表来聚合结果
        q_counts = {q_label: 0 for q_label in ordered_q_labels}
        for region_name, q_label in label_mapping.items():
            q_counts[q_label] += counts.get(region_name, 0)
        
        result_row = {'Scenario': config['label'], **q_counts}
        results.append(result_row)
        
    if not results:
        print("没有可供分析的数据。")
        return
        
    # 2. 创建并打印统计结果的DataFrame
    stats_df = pd.DataFrame(results).set_index('Scenario')
    print("\n--- Q1-Q6 统计分析结果 (仅针对显著变化的岛屿) ---")
    print(stats_df[ordered_q_labels])
    print("-" * 50)
    
    # 3. 准备绘图数据
    stats_df_melted = stats_df.reset_index().melt(
        id_vars='Scenario',
        var_name='Category',
        value_name='Number of Islands'
    )
    
    palette = {config['label']: config['color'] for config in scenario_config.values()}
    
    # 4. 创建竖向条形图
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    
    sns.barplot(
        data=stats_df_melted,
        x='Category',
        y='Number of Islands',
        hue='Scenario',
        palette=palette,
        ax=ax,
        legend=False,
        order=ordered_q_labels
    )
    
    # 5. 美化图表
    ax.set_ylabel('Number of Islands', fontsize=28)
    ax.set_xlabel('Category', fontsize=28)
    ax.spines[['right', 'top']].set_visible(False)
    ax.tick_params(axis='y', labelsize=28)
    ax.tick_params(axis='x', labelsize=28)
    
    fig.tight_layout()
    plt.show()
    
    # 6. 单独创建图例
    fig_legend, ax_legend = plt.subplots(figsize=(10, 2), dpi=300)
    ax_legend.axis('off')
    legend_elements = [plt.Rectangle((0,0),1,1, facecolor=config['color'], label=config['label']) 
                       for config in scenario_config.values()]
    legend = ax_legend.legend(handles=legend_elements, loc='center', ncol=len(scenario_config), fontsize=20, frameon=False)
    for text in legend.get_texts():
        text.set_fontweight('normal')
    fig_legend.tight_layout()
    plt.show()

# <<< MODIFICATION END: Replaced statistics function >>>


# --- 核心绘图函数（修改版本） ---
def plot_four_scenario_changes(df, base_scenario, final_scenario, median_breakeven, zoom_config, scenario_config, change_threshold=0):
    """
    筛选出在首尾情景间有显著变化的岛屿，绘制它们在四个情景中的位置，并进行统计分析。
    """
    if df.empty:
        print("数据为空，跳过绘图。")
        return

    df_base = df[df['scenario'] == base_scenario]
    df_final = df[df['scenario'] == final_scenario]
    
    if df_base.empty or df_final.empty:
        print(f"缺少 {base_scenario} 或 {final_scenario} 的数据，无法计算变化。")
        return

    change_data = calculate_position_change(df_base, df_final)
    significant_changes = change_data[change_data['position_change'] < change_threshold]
    
    if significant_changes.empty:
        print(f"在阈值 {change_threshold} 下未发现显著变化的岛屿。")
        return
        
    significant_island_ids = significant_changes['island_id'].unique()
    print(f"发现 {len(significant_island_ids)} 个有显著变化的岛屿（在 {base_scenario} 和 {final_scenario} 之间）。")

    df_plot = df[df['island_id'].isin(significant_island_ids)].copy()

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    divider = make_axes_locatable(ax)
    ax_top = divider.append_axes("top", size="15%", pad=0.05, sharex=ax)
    ax_right = divider.append_axes("right", size="15%", pad=0.05, sharey=ax)

    plt.setp(ax_top.get_xticklabels(), visible=False)
    plt.setp(ax_right.get_yticklabels(), visible=False)

    # Invert x-axis first
    ax.invert_xaxis()
    
    # Set axis limits
    ax.set_xlim(1.5, -0.2)
    ax.set_ylim(-0.2, 3.5)
    
    # <<< MODIFICATION: Draw background colors >>>
    draw_six_region_backgrounds(ax, median_breakeven, ax.get_xlim(), ax.get_ylim())
    
    for scenario_name, config in scenario_config.items():
        scenario_df = df_plot[df_plot['scenario'] == scenario_name]
        if not scenario_df.empty:
            ax.scatter(
                scenario_df['tariff_breakeven'], 
                scenario_df['tariff_affordable'],
                color=config['color'], alpha=config['alpha'], s=config['size'],
                edgecolors='white', linewidth=0.3, label=config['label'],
                zorder=config['zorder']
            )

    print(f"使用output_0基准中位数: median_breakeven={median_breakeven:.3f}")

    ax.set_xlabel('Cost Recovery Electricity Price (USD/kWh)', labelpad=18)
    ax.set_ylabel('Affordable Electricity Price (USD/kWh)', labelpad=18)
    
    ax.spines[['right', 'top']].set_visible(False)
    # <<< MODIFICATION: Update median lines to use single median >>>
    ax.axvline(x=median_breakeven, color='black', linestyle='--', linewidth=1, zorder=4)
    ax.axhline(y=median_breakeven, color='black', linestyle='--', linewidth=1, zorder=4)
    
    lim_min = min(ax.get_xlim()[0], ax.get_ylim()[0])
    lim_max = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lim_min, lim_max], [lim_min, lim_max], color='#C44E52', linestyle='--',
            alpha=1, linewidth=1.5, label='Break-even line', zorder=3)
    ax.grid(False)
    
    all_scenario_data = df_plot[df_plot['scenario'].isin(scenario_config.keys())]

    if not all_scenario_data.empty and len(all_scenario_data) > 10:
        try:
            x_data = all_scenario_data['tariff_breakeven'].values
            if len(x_data) > 1:
                kde_x = stats.gaussian_kde(x_data)
                x_range = np.linspace(ax.get_xlim()[1], ax.get_xlim()[0], 100)
                density_x = kde_x(x_range)
                ax_top.plot(x_range, density_x, color='#1a3563', linewidth=2, alpha=0.8)
                ax_top.fill_between(x_range, density_x, alpha=0.3, color='#1a3563')
                ax_top.set_xlim(ax.get_xlim())
                ax_top.spines[['right', 'top', 'left']].set_visible(False)
                ax_top.set_yticks([])
                ax_top.set_facecolor('none')

            y_data = all_scenario_data['tariff_affordable'].values
            if len(y_data) > 1:
                kde_y = stats.gaussian_kde(y_data)
                y_range = np.linspace(ax.get_ylim()[0], ax.get_ylim()[1], 100)
                density_y = kde_y(y_range)
                ax_right.plot(density_y, y_range, color='#1a3563', linewidth=2, alpha=0.8)
                ax_right.fill_betweenx(y_range, density_y, alpha=0.3, color='#1a3563')
                ax_right.set_ylim(ax.get_ylim())
                ax_right.spines[['right', 'top', 'bottom']].set_visible(False)
                ax_right.set_xticks([])
                ax_right.set_facecolor('none')

        except Exception as e:
            print(f"密度图绘制失败: {e}")
    
    fig.tight_layout()
    plt.show()

    # <<< MODIFICATION: Call the 6-region statistics function >>>
    q_label_mapping = {
        SIX_REGION_NAMES[0]: "Q1",
        SIX_REGION_NAMES[1]: "Q2",
        SIX_REGION_NAMES[2]: "Q3",
        SIX_REGION_NAMES[3]: "Q4",
        SIX_REGION_NAMES[4]: "Q5",
        SIX_REGION_NAMES[5]: "Q6"
    }
    analyze_and_plot_statistics(df_plot, scenario_config, median_breakeven, label_mapping=q_label_mapping)


# ===== 主程序 =====
if __name__ == '__main__':
    # --- 1. 数据加载和预处理 ---
    try:
        ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")
        ipcc_regions.sindex
        df['ipcc_region'] = df.apply(lambda row: assign_ipcc_region(row['lat'], row['lon'], ipcc_regions), axis=1)
    except Exception as e:
        print(f"无法加载 geojson 文件: {e}. 将无法分配IPCC区域。")
        df['ipcc_region'] = 'Unknown'

    # --- 2. 绘图配置 ---
    ZOOM_BOX = {
        'x_min': -0.1, 'x_max': 1.3,
        'y_min': -0.1, 'y_max': 2.7
    }

    SCENARIO_CONFIG = {
        'output_2050': {
            'label': 'Climate Stress', 'color': '#C44E52', 'alpha': 0.6,
            'size': 20, 'zorder': 2
        },
        'output_future_2030': {
            'label': 'TP2030', 'color': '#73a2c6', 'alpha': 0.7,
            'size': 25, 'zorder': 3
        },
        'output_future_2040': {
            'label': 'TP2040', 'color': '#3a67a3', 'alpha': 0.8,
            'size': 35, 'zorder': 4
        },
        'output_future_2050': {
            'label': 'TP2050', 'color': '#1a3563', 'alpha': 0.8,
            'size': 40, 'zorder': 5
        }
    }

    # --- 3. 筛选有效区域 ---
    MIN_ISLANDS_PER_REGION = 10
    region_counts = df['ipcc_region'].value_counts()
    valid_regions = region_counts[region_counts > MIN_ISLANDS_PER_REGION].index.tolist()

    scenarios_to_plot = list(SCENARIO_CONFIG.keys())
    df_filtered = df[(df['scenario'].isin(scenarios_to_plot)) & (df['ipcc_region'].isin(valid_regions))].copy()

    print(f"\n--- 正在对比四个情景：Climate Stress、TP2030、TP2040、TP2050 ---")
    
    # <<< MODIFICATION: Calculate only the single median_breakeven >>>
    # --- 4. 从原始df中计算中位数 ---
    df_output0 = df[df['scenario'] == 'output_0']
    median_b = df_output0['tariff_breakeven'].median()
    
    # <<< MODIFICATION: Update the function call >>>
    # --- 5. 调用绘图函数 ---
    if not df_filtered.empty:
        plot_four_scenario_changes(
            df=df_filtered,
            base_scenario='output_2050',
            final_scenario='output_future_2050',
            median_breakeven=median_b,
            zoom_config=ZOOM_BOX,
            scenario_config=SCENARIO_CONFIG,
            change_threshold=-0.05
        )
    else:
        print("没有足够的数据用于绘图。")

In [ ]:
from scipy import stats

# --- 修改后：字体变大的 Nature 风格图表设置 ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 16,        # 基础字体大小 (原为 12)
    'axes.labelsize': 16,   # 坐标轴标签 (原为 12)
    'axes.titlesize': 16,   # 坐标轴标题 (原为 12)
    'xtick.labelsize': 14,  # X轴刻度 (原为 12)
    'ytick.labelsize': 14,  # Y轴刻度 (原为 12)
    'legend.fontsize': 14,  # 图例 (原为 10)
    'figure.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})


def assign_ipcc_region(lat, lon, ipcc_regions_gdf):
    """将岛屿坐标分配到IPCC区域"""
    point = Point(lon, lat)
    possible_matches_index = list(ipcc_regions_gdf.sindex.intersection(point.bounds))
    possible_matches = ipcc_regions_gdf.iloc[possible_matches_index]
    precise_matches = possible_matches[possible_matches.contains(point)]
    if not precise_matches.empty:
        return precise_matches.iloc[0]['Acronym']
    return 'Unknown'

def calculate_position_change(df_base, df_compare):
    """计算两个情景之间的位置变化"""
    # 合并数据，基于岛屿ID - 使用固定的列名，与fig4.1.ipynb一致
    merged = pd.merge(df_base[['island_id', 'tariff_breakeven', 'tariff_affordable', 'ipcc_region']],
                      df_compare[['island_id', 'tariff_breakeven', 'tariff_affordable', 'ipcc_region']],
                      on='island_id', suffixes=('_base', '_compare'))

    # 只有成本回收电价会变，因此变化的距离只需要相减
    merged['position_change'] = merged['tariff_breakeven_compare'] - merged['tariff_breakeven_base']
    return merged

def create_raincloud_plot(data_2030, data_2040, data_2050, ax, region_name, color_2030, color_2040, color_2050):
    """
    创建改进的云雨图：三个scenario占据上中下三个区域，每个区域内从下到上是点-箱线图-半小提琴图
    
    Args:
        data_2030, data_2040, data_2050: 三个情景的变化值数据
        ax: matplotlib轴对象
        region_name: IPCC区域名称
        color_2030, color_2040, color_2050: 三个情景的颜色
    """
    # 计算数据范围
    all_data = np.concatenate([data_2030, data_2040, data_2050])
    if len(all_data) == 0:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{region_name}', pad=10, fontweight='bold',fontsize=14)
        return
        
    x_min, x_max = all_data.min(), all_data.max()
    x_range = x_max - x_min if x_max != x_min else 1
    x_padding = x_range * 0.1
    
    # 设置y轴位置（分为三个区域）
    y_positions = {
        2030: {'points': 0.80, 'box': 0.85, 'violin': 0.90},
        2040: {'points': 0.45, 'box': 0.50, 'violin': 0.55},
        2050: {'points': 0.10, 'box': 0.15, 'violin': 0.20}
    }
    
    np.random.seed(42)  # 确保结果可重现
    
    # 绘制三个时间段的数据
    for year, data, color in [(2030, data_2030, color_2030), 
                             (2040, data_2040, color_2040),
                             (2050, data_2050, color_2050)]:
        if len(data) > 0:
            pos = y_positions[year]
            
            # 1. 散点图
            y_jitter = np.random.normal(pos['points'], 0.015, len(data))
            ax.scatter(data, y_jitter, alpha=0.9, s=15, 
                      color=color, edgecolors='white', linewidth=0.5, zorder=3)
            
            # 2. 箱线图
            if len(data) > 1:
                q25, q50, q75 = np.percentile(data, [25, 50, 75])
                iqr = q75 - q25
                whisker_low = max(data.min(), q25 - 1.5 * iqr)
                whisker_high = min(data.max(), q75 + 1.5 * iqr)
                
                box_height = 0.06
                ax.add_patch(plt.Rectangle((q25, pos['box'] - box_height/2), q75 - q25, box_height,
                                          facecolor=color, alpha=0.9, zorder=2))
                ax.plot([q50, q50], [pos['box'] - box_height/2, pos['box'] + box_height/2], 
                       color='black', linewidth=2, zorder=4)
                ax.plot([whisker_low, q25], [pos['box'], pos['box']], color='black', linewidth=1, zorder=2)
                ax.plot([q75, whisker_high], [pos['box'], pos['box']], color='black', linewidth=1, zorder=2)
                ax.plot([whisker_low, whisker_low], [pos['box'] - box_height/4, pos['box'] + box_height/4], 
                       color='black', linewidth=1, zorder=2)
                ax.plot([whisker_high, whisker_high], [pos['box'] - box_height/4, pos['box'] + box_height/4], 
                       color='black', linewidth=1, zorder=2)
            
            # 3. 半小提琴图
            if len(data) > 3:
                try:
                    kde = stats.gaussian_kde(data)
                    x_kde = np.linspace(x_min - x_padding, x_max + x_padding, 100)
                    density = kde(x_kde)
                    density_scaled = density / density.max() * 0.12
                    
                    ax.fill_between(x_kde, pos['violin'], pos['violin'] + density_scaled, 
                                  alpha=0.9, color=color, zorder=1)
                    ax.plot(x_kde, [pos['violin']] * len(x_kde), color=color, linewidth=1, alpha=0.8, zorder=1)
                except:
                    pass
    
    # 设置轴和标签
    ax.set_ylim(-0.02, 1.02)
    ax.set_yticks([0.15, 0.50, 0.85])
    ax.set_yticklabels(['TP2050', 'TP2040', 'TP2030'], fontweight='bold')
    ax.set_title(f'{region_name}', fontsize=14, pad=6, fontweight='bold')
    ax.grid(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set_xlim(x_min - x_padding, x_max + x_padding)

# 主程序：创建多子图云雨图
if __name__ == '__main__':
    # 首先加载IPCC区域数据并分配区域，与fig4.1.ipynb保持一致
    try:
        from shapely.geometry import Point
        import geopandas as gpd
        ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")
        ipcc_regions.sindex  # 确保空间索引存在
        df['ipcc_region'] = df.apply(lambda row: assign_ipcc_region(row['lat'], row['lon'], ipcc_regions), axis=1)
    except Exception as e:
        print(f"无法加载 geojson 文件: {e}. 将无法分配IPCC区域。")
        df['ipcc_region'] = 'Unknown'

    # 筛选有效区域（与fig4.1.ipynb相同的逻辑）
    MIN_ISLANDS_PER_REGION = 5
    region_counts = df['ipcc_region'].value_counts()
    valid_regions = region_counts[region_counts > MIN_ISLANDS_PER_REGION].index.tolist()
    
    print(f"有效IPCC区域数量: {len(valid_regions)}")
    print(f"有效区域列表: {valid_regions}")
    
    # 计算变化数据 - 修改为以output_0为基准
    df_base_output0 = df[(df['scenario'] == 'output_0') & (df['ipcc_region'].isin(valid_regions))].copy()
    df_compare_2030 = df[(df['scenario'] == 'output_future_2030') & (df['ipcc_region'].isin(valid_regions))].copy()
    df_compare_2040 = df[(df['scenario'] == 'output_future_2040') & (df['ipcc_region'].isin(valid_regions))].copy()
    df_compare_2050 = df[(df['scenario'] == 'output_future_2050') & (df['ipcc_region'].isin(valid_regions))].copy()

    # 检查数据可用性
    print(f"Base scenario (output_0) data: {len(df_base_output0)} records")
    print(f"Future 2030 scenario data: {len(df_compare_2030)} records")  
    print(f"Future 2040 scenario data: {len(df_compare_2040)} records")
    print(f"Future 2050 scenario data: {len(df_compare_2050)} records")
    
    if len(df_base_output0) == 0:
        print("错误：没有找到基准情景 (output_0) 的数据")
        exit()

    # 计算变化（相对于output_0的变化）
    change_2030 = pd.DataFrame()
    change_2040 = pd.DataFrame() 
    change_2050 = pd.DataFrame()
    
    if len(df_compare_2030) > 0:
        change_2030 = calculate_position_change(df_base_output0, df_compare_2030)
    if len(df_compare_2040) > 0:
        change_2040 = calculate_position_change(df_base_output0, df_compare_2040) 
    if len(df_compare_2050) > 0:
        change_2050 = calculate_position_change(df_base_output0, df_compare_2050)
    
    # 筛选有显著变化的区域
    significant_2030 = change_2030[change_2030['position_change'] < -0.03] if len(change_2030) > 0 else pd.DataFrame()
    significant_2040 = change_2040[change_2040['position_change'] < -0.03] if len(change_2040) > 0 else pd.DataFrame()
    significant_2050 = change_2050[change_2050['position_change'] < -0.03] if len(change_2050) > 0 else pd.DataFrame()
    
    # 获取在任一scenario中都有显著变化的区域
    regions_2030 = set(significant_2030['ipcc_region_base'].unique()) if len(significant_2030) > 0 else set()
    regions_2040 = set(significant_2040['ipcc_region_base'].unique()) if len(significant_2040) > 0 else set()
    regions_2050 = set(significant_2050['ipcc_region_base'].unique()) if len(significant_2050) > 0 else set()
    common_regions = list(regions_2030.union(regions_2040).union(regions_2050))
    
    print(f"发现在至少一个scenario中有显著变化的区域（相对于output_0基准）: {len(common_regions)} 个")
    print(f"区域列表: {common_regions}")
    
    if len(common_regions) == 0:
        print("没有发现有显著变化的区域")
    else:
        # 计算子图布局
        n_regions = len(common_regions)
        n_cols = min(4, n_regions)  # 最多4列
        n_rows = (n_regions + n_cols - 1) // n_cols  # 向上取整
        
        # 创建图形
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows), dpi=300)
        if n_regions == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes.flatten()
        else:
            axes = axes.flatten()
        
        # 定义颜色
        color_2030 = '#73a2c6'  # Blue
        color_2040 = '#3a67a3'  # Orange
        color_2050 = '#1a3563'  # Red
        
        # 为每个区域创建子图（修改为三个时间段）
        for i, region in enumerate(common_regions):
            ax = axes[i]
            
            # 获取该区域在三个scenario下的变化数据（相对于output_0基准）
            region_data_2030 = change_2030[change_2030['ipcc_region_base'] == region]['position_change'].values if len(change_2030) > 0 else np.array([])
            region_data_2040 = change_2040[change_2040['ipcc_region_base'] == region]['position_change'].values if len(change_2040) > 0 else np.array([])
            region_data_2050 = change_2050[change_2050['ipcc_region_base'] == region]['position_change'].values if len(change_2050) > 0 else np.array([])
            
            # 创建云雨图
            create_raincloud_plot(region_data_2030, region_data_2040, region_data_2050, ax, 
                                region, color_2030, color_2040, color_2050)
        
        # 隐藏多余的子图
        for i in range(n_regions, len(axes)):
            axes[i].set_visible(False)
        
        # 为所有子图添加统一的x轴标签
        for i in range(n_regions):
            if i >= (n_rows-1) * n_cols:  # 最后一行
                axes[i].set_xlabel('Electricity Price Change (USD/kWh)')
        
        # 添加图例
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        legend_elements = [
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_2030, 
                  markersize=10, label='TP2030', alpha=0.8),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_2040, 
                  markersize=10, label='TP2040', alpha=0.8),
            Line2D([0], [0], marker='o', color='w', markerfacecolor=color_2050, 
                  markersize=10, label='TP2050', alpha=0.8),
            # Patch(facecolor='lightgray', alpha=0.6, label='Box Plot'),
            # Patch(facecolor='lightgray', alpha=0.4, label='Density (Half-Violin)'),
            Line2D([0], [0], color='black', linewidth=2, label='Median')
        ]
        # fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.95), ncol=7, fontsize=14)
        
        # Create separate legend figure
        fig_legend = plt.figure(figsize=(10, 1.5), dpi=300)
        ax_legend = fig_legend.add_subplot(111)
        ax_legend.axis('off')

        legend = ax_legend.legend(handles=legend_elements,
                                loc='center',
                                frameon=False,
                                fontsize=16,
                                ncol=7,
                                columnspacing=1.2)

        # fig_legend.savefig('fig5_raincloud_legend.png',
        #                  dpi=300,
        #                  bbox_inches='tight',
        #                  pad_inches=0.05,
        #                  transparent=True)
        plt.show(fig_legend)
        # plt.close(fig_legend)
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.90)
        plt.show()
        
        # 打印统计信息
        print(f"\n=== 区域变化统计（相对于output_0基准） ===")
        for region in common_regions:
            data_2030 = change_2030[change_2030['ipcc_region_base'] == region]['position_change'].values if len(change_2030) > 0 else np.array([])
            data_2040 = change_2040[change_2040['ipcc_region_base'] == region]['position_change'].values if len(change_2040) > 0 else np.array([])
            data_2050 = change_2050[change_2050['ipcc_region_base'] == region]['position_change'].values if len(change_2050) > 0 else np.array([])
            
            print(f"\n{region}:")
            if len(data_2030) > 0:
                print(f"  TP2030: {len(data_2030)} islands, median = {np.median(data_2030):.3f}, max = {np.max(data_2030):.3f}")
            if len(data_2040) > 0:
                print(f"  TP2040: {len(data_2040)} islands, median = {np.median(data_2040):.3f}, max = {np.max(data_2040):.3f}")
            if len(data_2050) > 0:
                print(f"  TP2050: {len(data_2050)} islands, median = {np.median(data_2050):.3f}, max = {np.max(data_2050):.3f}")

In [ ]:
# 7. --- 基于有明显变化岛屿的点密度世界地图 (output_2050 vs output_future_2050) ---

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter

print("开始创建点密度世界地图 (对比 Climate Stress vs TP2050)...")

# --- MODIFIED: 修改对比情景 ---
# 计算岛屿变化数据
df_base = df[(df['scenario'] == 'output_2050')].copy()
df_compare = df[(df['scenario'] == 'output_future_2050')].copy()

# 使用现有的calculate_position_change函数
change_data = calculate_position_change(df_base, df_compare)
print(change_data['position_change'].describe())
# --- MODIFIED: 将显著变化定义为成本显著降低（负值） ---
change_threshold = -0.015
significant_changes = change_data[change_data['position_change'] < change_threshold]

print(f"发现 {len(significant_changes)} 个成本显著降低的岛屿")

if len(significant_changes) > 0:
    # 获取有显著变化岛屿的位置信息
    # 注意：我们从 df_base 获取原始位置和其他信息
    significant_islands = df_base[df_base['island_id'].isin(significant_changes['island_id'])].copy()
    
    # 将变化量信息合并到岛屿数据中
    significant_islands = significant_islands.merge(
        significant_changes[['island_id', 'position_change']], 
        on='island_id', how='left'
    )
    
    print(f"成本显著降低岛屿的IPCC区域分布:")
    region_counts = significant_islands['ipcc_region'].value_counts()
    print(region_counts)
    
    # 读取IPCC区域地理数据
    ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")

    # 创建世界地图
    fig = plt.figure(figsize=(14, 8.5), dpi=300)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    ax.set_facecolor("#FFFFFF")
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
    
    # 计算每个IPCC区域的显著变化岛屿数量
    if not region_counts.empty:
        vmin, vmax = region_counts.min(), region_counts.max()
        norm = Normalize(vmin=0, vmax=vmax)  # 从0开始，确保没有岛屿的区域为白色
        
        # --- MODIFIED: 创建自定义蓝色渐变色彩映射，代表积极变化 ---
        colors = ['#FFFFFF', '#D6EAF8', '#A9CCE3', '#5DADE2', '#3498DB', '#2E86C1', '#1B4F72'] # 白色到深蓝色
        blue_colormap = LinearSegmentedColormap.from_list('blue_gradient', colors, N=256)
        
        # 为每个IPCC区域着色
        for _, region_row in ipcc_regions.iterrows():
            region_code = region_row['Acronym']
            
            if region_code in region_counts.index:
                # 该区域有显著变化的岛屿
                island_count = region_counts[region_code]
                color = blue_colormap(norm(island_count))
                alpha = 0.7  # 有数据的区域更不透明
                
                # 绘制填充的区域
                ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                                  facecolor=color,
                                  edgecolor='white',
                                  linewidth=0.5,
                                  alpha=alpha,
                                  zorder=1)
            else:
                # 该区域没有显著变化的岛屿，保持浅色
                ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                                  facecolor="#FFFFFF", 
                                  edgecolor="white",
                                  linewidth=0.3,
                                  alpha=0.5,
                                  zorder=1)
    
    # 在地图上标注显著变化的岛屿点 - 根据变化量调整点的大小
    if not significant_islands.empty:
        # --- MODIFIED: 根据成本降低的幅度（绝对值）来计算点大小 ---
        change_values = np.abs(significant_islands['position_change'].values)
        min_change = change_values.min()
        max_change = change_values.max()
        
        # 设置点大小范围
        min_size = 50   # 最小点大小
        max_size = 500  # 最大点大小
        
        # 线性缩放点大小：变化幅度越大，点越大
        if max_change > min_change:
            normalized_changes = (change_values - min_change) / (max_change - min_change)
            point_sizes = min_size + normalized_changes * (max_size - min_size)
        else:
            point_sizes = np.full(len(change_values), (min_size + max_size) / 2)
        
        # --- MODIFIED: 使用深蓝色圆点标记 ---
        scatter = ax.scatter(significant_islands['lon'], significant_islands['lat'],
                             transform=ccrs.PlateCarree(),
                             c="#1A3563",  # 深蓝色点
                             s=point_sizes,  # 根据变化幅度调整点大小
                             alpha=0.9,
                             edgecolors='white',
                             linewidth=0.5,
                             zorder=3)
        
        print(f"点大小范围: {point_sizes.min():.1f} - {point_sizes.max():.1f}")
        print(f"成本降低量范围: {min_change:.3f} - {max_change:.3f}")
    
    # 设置视图范围，排除南极洲
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    
    # (可选) 添加颜色条图例
    # if not region_counts.empty:
    #     sm = cm.ScalarMappable(norm=norm, cmap=blue_colormap)
    #     sm.set_array([])
    #     cbar = plt.colorbar(sm, ax=ax, orientation='vertical', 
    #                         pad=0.02, shrink=0.6, aspect=20)
    #     cbar.set_label('Number of Islands with Significant Cost Reduction', 
    #                    rotation=270, labelpad=20, fontsize=12)
    #     cbar.ax.tick_params(labelsize=10)
    
    # 去除坐标轴刻度和边框
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 去除所有边距，只显示图像内容
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis('off')
    plt.show()
    
    # 打印统计摘要
    print(f"\n=== 世界地图统计摘要 (Climate Stress vs TP2050) ===")
    print(f"总共标记了 {len(significant_islands)} 个成本显著降低的岛屿")
    print(f"涉及 {len(region_counts)} 个IPCC区域")
    if not region_counts.empty:
        print(f"变化最多的区域: {region_counts.index[0]} ({region_counts.iloc[0]} 个岛屿)")
    
else:
    print("在 Climate Stress 和 TP2050 情景之间没有发现成本显著降低的岛屿，无法创建地图。")